# QueryGenie — Tier-2 reproduction & Week-7 ablations (CodeS on Spider dev)

Team 21 · anchor paper CodeS (arXiv:2402.16347) · **Runtime → Change runtime type → T4 GPU** before running.

Every cell fetches the same self-contained script (`scripts/colab_eval_tier2.py` in the repo, mirrored on a gist). It downloads Spider + the model once per runtime, evaluates all 1034 dev examples with the authors' full database prompt and candidate-selection rule, and prints a `COPY THIS BACK:` line to paste into `results/reproduction_log.csv`.

Env switches: `QG_MODEL` (default `seeklhy/codes-1b-spider`), `QG_RUN_ID`, `QG_ABLATE` (`no_values`, `no_content`, `no_select`, `no_fk`).

## 0 · Check GPU

In [ ]:
!nvidia-smi -L
import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)

## 1 · Faithful reproduction — CodeS-1B (Table 5 target: 77.9% EX)
T2-002 = full prompt, beam 0 only → 71.7%. T2-003 = + authors' first-executable-candidate rule → 73.8%. Re-run only if needed.

In [ ]:
import os, urllib.request; os.environ["QG_RUN_ID"]="T2-003"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

## 2 · Week 7 — ablation study (baseline = T2-003, 73.8%)
Remove one component at a time; the drop shows how much it contributes.

### AB-003 · no representative column values in the schema

In [ ]:
import os, urllib.request; os.environ["QG_RUN_ID"]="AB-003"; os.environ["QG_ABLATE"]="no_values"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

### AB-004 · no matched contents (value retriever off)

In [ ]:
import os, urllib.request; os.environ["QG_RUN_ID"]="AB-004"; os.environ["QG_ABLATE"]="no_content"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

### AB-005 · no foreign-keys block

In [ ]:
import os, urllib.request; os.environ["QG_RUN_ID"]="AB-005"; os.environ["QG_ABLATE"]="no_fk"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

### AB-006 · beam 0 only (candidate selection off) — re-measures T2-002 with the same code path

In [ ]:
import os, urllib.request; os.environ["QG_RUN_ID"]="AB-006"; os.environ["QG_ABLATE"]="no_select"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

## 3 · Larger model — CodeS-3B (Table 5 target: 83.4% EX)
Same pipeline, bigger checkpoint (~6 GB fp16, fits a T4). ~1.5–2 h.

In [ ]:
import os, urllib.request; os.environ["QG_RUN_ID"]="T2-004"; os.environ["QG_MODEL"]="seeklhy/codes-3b-spider"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())